In [1]:
import os
import os.path as op

# Remove any preset backend
os.environ.pop("MPLBACKEND", None)

# importing the tools we'll use throughout the rest of the script
# sys is system tools, should already be installed
import sys
import json

# pandas is a dataframe-managing library and it's the absolute coolest
import pandas as pd

# numpy is short for "numerical python" and it does math
import numpy as np

# seaborn is a plotting library named after a character from West Wing
# it's kind of like python's ggplot
import seaborn as sns

# nibabel handles nifti images
import nibabel as nib

# os is more system tools, should also already be installed
# we're importing tools for verifying and manipulating file paths/directories
from os.path import join, exists, isdir
from os import makedirs

# nilearn makes the best brain plots
# and their documentation/examples are so, so handy
# https://nilearn.github.io/stable/auto_examples/01_plotting/index.html
from nilearn import plotting, surface, datasets
from nilearn.plotting.cm import _cmap_d as nilearn_cmaps
from nilearn.maskers import NiftiMasker
from nilearn.image import threshold_img, index_img
from nilearn.plotting import plot_stat_map

# matplotlib is the backbone of most python plotting
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

# gridspec helps us put lots of panels on one figure
from PIL import Image

from neuromaps import transforms
from neuromaps.datasets import fetch_fslr\

from nimare.decode.continuous import CorrelationDecoder
from nimare.transforms import p_to_z

from surfplot import Plot


from gradec.fetcher import _fetch_features, _fetch_frequencies, _fetch_classification
from gradec.utils import _decoding_filter
from gradec.utils import _zero_medial_wall
from netneurotools import stats

from gradec.plot import plot_radar, plot_cloud




In [8]:
CMAP = nilearn_cmaps["cold_hot"]
CMAP = "Spectral_r"

In [9]:
# # Define the number of colors in the colormap
# num_colors = 1000

# # Define anchor points for key colors using hex values
# colors = [
#     "#171744",  # Dark Blue
#     "#322ce8",
#     "#1cc5e3",  # light blue
#     "#5fbfb1",
#     "#a1b880",
#     "#a1b880",
#     "#E4B24E",  # Yellow (adjusted)
#     "#E4B24E",  # Yellow (adjusted)
#     "#ed774d",  # Orange
#     # "#f63b4d",  # Orange
#     "#ff004c",  # Red
# ]
# # Create a smooth interpolation for the custom colormap
# CMAP = LinearSegmentedColormap.from_list("custom_cmap", colors, N=num_colors)

# # Plot the color gradient
# plt.figure(figsize=(8, 2))
# plt.imshow([list(range(num_colors))], aspect="auto", cmap=CMAP)
# plt.gca().set_visible(False)
# plt.colorbar(orientation="horizontal")
# plt.show()

In [10]:
def trim_image(img=None, tol=1, fix=True):
    if fix:
        mask = img != tol
    else:
        mask = img <= tol
    if img.ndim == 3:
        mask = mask.any(2)
    mask0, mask1 = mask.any(0), mask.any(1)
    return img[np.ix_(mask1, mask0)]

In [11]:
def plot_vol(
    nii_img_thr, threshold, mask_contours=None, vmax=6, alpha=1, cmap=CMAP, dim=-0.45
):
    template = datasets.load_mni152_template(resolution=1)

    display_modes = ["x", "y", "z"]
    fig = plt.figure(figsize=(5, 5))
    fig.subplots_adjust(
        left=None, bottom=None, right=None, top=None, wspace=None, hspace=None
    )
    gs = GridSpec(2, 2, figure=fig)

    for dsp_i, display_mode in enumerate(display_modes):
        if display_mode == "z":
            ax = fig.add_subplot(gs[:, 1], aspect="equal")
            colorbar = True
        else:
            ax = fig.add_subplot(gs[dsp_i, 0], aspect="equal")
            colorbar = False

        display = plot_stat_map(
            nii_img_thr,
            bg_img=template,
            black_bg=False,
            draw_cross=False,
            annotate=True,
            alpha=alpha,
            cmap=cmap,
            threshold=threshold,
            colorbar=colorbar,
            display_mode=display_mode,
            cut_coords=1,
            vmax=vmax,
            axes=ax,
            dim=dim,  # Adjusted dimming factor
        )
        if mask_contours:
            display.add_contours(mask_contours, levels=[0.5], colors="black")

    return fig

In [12]:
def plot_surf(nii_img_thr, mask_contours=None, vmax=8, cmap=CMAP):
    map_lh, map_rh = transforms.mni152_to_fslr(nii_img_thr, fslr_density="32k")
    map_lh, map_rh = _zero_medial_wall(
        map_lh,
        map_rh,
        space="fsLR",
        density="32k",
    )
    # midthickness

    surfaces = fetch_fslr(density="32k")
    lh, rh = surfaces["inflated"]
    sulc_lh, sulc_rh = surfaces["sulc"]

    p = Plot(surf_lh=lh, surf_rh=rh, layout="grid")
    p.add_layer({"left": sulc_lh, "right": sulc_rh}, cmap="binary_r", cbar=False)
    p.add_layer(
        {"left": map_lh, "right": map_rh},
        cmap=cmap,
        cbar=False,
        color_range=(-vmax, vmax),
    )
    if mask_contours:
        mask_lh, mask_rh = transforms.mni152_to_fslr(mask_contours, fslr_density="32k")
        mask_lh, mask_rh = _zero_medial_wall(
            mask_lh,
            mask_rh,
            space="fsLR",
            density="32k",
        )
        mask_arr_lh = mask_lh.agg_data()
        mask_arr_rh = mask_rh.agg_data()
        countours_lh = np.zeros_like(mask_arr_lh)
        countours_lh[mask_arr_lh != 0] = 1
        countours_rh = np.zeros_like(mask_arr_rh)
        countours_rh[mask_arr_rh != 0] = 1

        colors = [(0, 0, 0, 0)]
        contour_cmap = ListedColormap(colors, "regions", N=1)
        line_cmap = ListedColormap(["black"], "regions", N=1)
        p.add_layer(
            {"left": countours_lh, "right": countours_rh},
            cmap=line_cmap,
            as_outline=True,
            cbar=False,
        )
        p.add_layer(
            {"left": countours_lh, "right": countours_rh},
            cmap=contour_cmap,
            cbar=False,
        )

    return p.build()

In [13]:
data_dir = "/Users/chloehampson/Desktop/diva-emotion-dfc/dset/derivatives/caps/caps_masks"
out_dir = "/Users/chloehampson/Desktop/diva-emotion-dfc/dset/derivatives/figures"

In [14]:
DSET, MODEL = "neuroquery", "lda"
decoding_dir = "./decoding"
decoder_fn = op.join(decoding_dir, f"{MODEL}_{DSET}_decoder.pkl.gz")

decoder = CorrelationDecoder.load(decoder_fn)

features = _fetch_features(DSET, MODEL, data_dir=decoding_dir)
frequencies = _fetch_frequencies(DSET, MODEL, data_dir=decoding_dir)
classification, class_lst = _fetch_classification(DSET, MODEL, data_dir=decoding_dir)

In [15]:
mask_img = datasets.load_mni152_brain_mask(resolution=1)
masker = NiftiMasker(mask_img=mask_img)
masker = masker.fit()

In [16]:
cap_img = join(data_dir, "sub-Blossom_zscore-weighted-0_CAP_1_neg.nii.gz")

In [17]:
import glob

# Get all CAP files in the data directory
cap_files = glob.glob(join(data_dir, "*_CAP_*_*.nii.gz"))
cap_files.sort()

print(f"Found {len(cap_files)} CAP files:")
for f in cap_files[:5]:  # Show first 5 as example
    print(f"  {op.basename(f)}")
if len(cap_files) > 5:
    print(f"  ... and {len(cap_files) - 5} more files")



Found 26 CAP files:
  sub-Blossom_zscore-weighted-0_CAP_1_neg.nii.gz
  sub-Blossom_zscore-weighted-0_CAP_1_pos.nii.gz
  sub-Blossom_zscore-weighted-0_CAP_2_neg.nii.gz
  sub-Blossom_zscore-weighted-0_CAP_2_pos.nii.gz
  sub-Blossom_zscore-weighted-0_CAP_3_neg.nii.gz
  ... and 21 more files


In [ ]:
 # Process each CAP file

# Define significant results for plotting - only participants/episodes/CAPs with significant findings
sig_participant_data = {
    "Blossom": [("1", "pos"), ("2", "neg"), ("2", "pos")],
    "Bubbles": [("1", "pos"), ("2", "neg")],
    "Buttercup": [("3", "neg"), ("4", "pos")]
}


for cap_img in cap_files:
    filename = op.basename(cap_img)
    
    # Extract information from filename
    # Format: sub-{subject}_zscore-weighted-0_CAP_{cap_num}_{pos/neg}.nii.gz
    parts = filename.split('_')
    subject = parts[0].replace('sub-', '')
    cap_num = parts[3]  # CAP number
    test = parts[4].split('.')[0]  # 'pos' or 'neg'
    
    # Check if this file should be processed based on sig_participant_data
    if subject not in sig_participant_data:
        print(f"Skipping {filename} - subject {subject} not in significant results")
        continue
    
    if (cap_num, test) not in sig_participant_data[subject]:
        print(f"Skipping {filename} - CAP {cap_num} {test} not significant for {subject}")
        continue
    
    print(f"\nProcessing: {filename}")
    print(f"  Subject: {subject}, CAP: {cap_num}, Test: {test}")
    
    # Create output directory for this specific analysis
    fig_save_dir = join(out_dir, f"sub-{subject}")
    makedirs(fig_save_dir, exist_ok=True)

    z_thresh = 0.00001

    # Continue with processing in the next cell...

   # Load and threshold images
    # Load and threshold images
    nii_img = nib.load(cap_img)
    
    # Apply threshold to create binary mask
    nii_thr_img = threshold_img(nii_img, threshold=z_thresh, two_sided=True)
    
    # Calculate effect sizes (Cohen's d)
    nii_arr = masker.transform(nii_img)
    
    # Create contour masks for visualization
    nii_thr_arr = masker.transform(nii_img)
    nii_contour_arr = np.zeros_like(nii_thr_arr)
    nii_contour_arr[(nii_thr_arr > z_thresh) | (nii_thr_arr < -z_thresh)] = 1
    nii_contour_img = masker.inverse_transform(nii_contour_arr)
    nii_contour_img_3d = index_img(nii_contour_img, 0)
    
    # Set visualization parameters
    vmax = round(np.max(np.abs(nii_thr_arr)), 2)
    vmax = 13 if vmax > 13 else vmax
    
    # Save thresholded images
    thresh_fn = op.join(fig_save_dir, f"sub-{subject}_CAP-{cap_num}_{test}_thresh.nii.gz")
    nib.save(nii_img, thresh_fn)
    print(f"  Saved thresholded nifti: {thresh_fn}")
    
    # Generate all visualizations
    print("  Generating visualizations...")
    
    # Volume plot
    vol_fig = plot_vol(nii_img, z_thresh, vmax=vmax, cmap=CMAP)
    vol_fig.savefig(op.join(fig_save_dir, f"sub-{subject}_CAP-{cap_num}_{test}_volume.png"), 
                    dpi=300, bbox_inches='tight')
    plt.close(vol_fig)
    
    # Surface plot
    surf_fig = plot_surf(nii_img, mask_contours=nii_contour_img_3d, vmax=vmax, cmap=CMAP)
    surf_fig.savefig(op.join(fig_save_dir, f"sub-{subject}_CAP-{cap_num}_{test}_surface.png"), 
                     dpi=300, bbox_inches='tight')
    plt.close(surf_fig)
    
    # Prepare data for decoding
    if test == "pos":
        # For positive test, keep only positive values
        nii_pos_arr = np.where(nii_arr > 0, nii_arr, 0)
        img_to_decode = masker.inverse_transform(nii_pos_arr)
    elif test == "neg":
        # For negative test, take absolute value of negative regions
        nii_neg_arr = np.where(nii_arr < 0, abs(nii_arr), 0)
        img_to_decode = masker.inverse_transform(nii_neg_arr)
    
    # Decode the image
    try:
        corrs_df = decoder.transform(img_to_decode)
        num_val = [int(lab.split("__")[1].split("_")[0]) for lab in corrs_df.index.to_list()]
        indices = np.argsort(num_val)
        corrs_df = corrs_df.iloc[indices]
        filtered_df, filtered_features, filtered_frequencies = _decoding_filter(
            corrs_df,
            features,
            classification,
            freq_by_topic=frequencies,
            class_by_topic=class_lst,
        )
        
        # Visualize decoding results
        corrs = filtered_df["r"].to_numpy()
        
        # Radar plot
        plot_radar(
            corrs,
            filtered_features,
            MODEL,
            cmap=CMAP,
            out_fig=op.join(fig_save_dir, f"sub-{subject}_CAP-{cap_num}_{test}_radar.png"),
        )
        
        # Word cloud plot
        plot_cloud(
            corrs,
            filtered_features,
            MODEL,
            width=10,
            height=5,
            frequencies=filtered_frequencies,
            cmap=CMAP,
            out_fig=op.join(fig_save_dir, f"sub-{subject}_CAP-{cap_num}_{test}_wordcloud.png"),
        )
        
        print(f"  Saved decoding plots for {subject} CAP-{cap_num} {test}")
        
    except Exception as e:
        print(f"  Error in decoding for {subject} CAP-{cap_num} {test}: {e}")
    
    print(f"  Completed processing: {filename}")

print("\nAll significant CAP files processed!")
        

Skipping sub-Blossom_zscore-weighted-0_CAP_1_neg.nii.gz - CAP 1 neg not significant for Blossom

Processing: sub-Blossom_zscore-weighted-0_CAP_1_pos.nii.gz
  Subject: Blossom, CAP: 1, Test: pos


/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  warnings.warn(
/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  warnings.warn(
/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You mig

  Saved thresholded nifti: /Users/chloehampson/Desktop/diva-emotion-dfc/dset/derivatives/figures/sub-Blossom/sub-Blossom_CAP-1_pos_thresh.nii.gz
  Generating visualizations...


/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  warnings.warn(
/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  warnings.warn(


  Saved decoding plots for Blossom CAP-1 pos
  Completed processing: sub-Blossom_zscore-weighted-0_CAP_1_pos.nii.gz

Processing: sub-Blossom_zscore-weighted-0_CAP_2_neg.nii.gz
  Subject: Blossom, CAP: 2, Test: neg


...

  Saved decoding plots for Buttercup CAP-3 neg
  Completed processing: sub-Buttercup_zscore-weighted-0_CAP_3_neg.nii.gz
Skipping sub-Buttercup_zscore-weighted-0_CAP_3_pos.nii.gz - CAP 3 pos not significant for Buttercup
Skipping sub-Buttercup_zscore-weighted-0_CAP_4_neg.nii.gz - CAP 4 neg not significant for Buttercup

Processing: sub-Buttercup_zscore-weighted-0_CAP_4_pos.nii.gz
  Subject: Buttercup, CAP: 4, Test: pos


/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  warnings.warn(
/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  warnings.warn(
/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You mig

  Saved thresholded nifti: /Users/chloehampson/Desktop/diva-emotion-dfc/dset/derivatives/figures/sub-Buttercup/sub-Buttercup_CAP-4_pos_thresh.nii.gz
  Generating visualizations...


/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  warnings.warn(
/Users/chloehampson/Desktop/habenula-abide-rsfc/.venv/lib/python3.9/site-packages/nilearn/maskers/nifti_masker.py:108: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  warnings.warn(


  Saved decoding plots for Buttercup CAP-4 pos
  Completed processing: sub-Buttercup_zscore-weighted-0_CAP_4_pos.nii.gz

All significant CAP files processed!
